## **07 -- Timing of the SMM indicators relative to the 24-hour prediction point**

**Purpose.** SMM is defined for the whole admission and diagnosis
codes carry no time. Procedure codes do carry a **date** (`procedures_icd.chartdate`, a calendar date with no clock time), so for the SMM
indicators that are procedures (ventilation, hysterectomy, transfusion, cardiac and others) we can say whether they happened before, around or
after the prediction point (admission time + 24 h). This notebook **counts** how many SMM events in the main population can be placed in time.
It trains no model, does not modify the database and does not touch any other notebook.

**Method.** For every eligible delivery hospitalization with SMM (`eligible_24h`, `is_delivery_v2`, `smm_any` of `mimiciv.predictors_24h_v2`), each
procedure-based SMM indicator is matched to `mimiciv.procedures_icd` with the same code patterns as the label (`mimiciv.smm_code_patterns_v2`)
and classified from its calendar date `D` and the prediction time `T = admittime + 24 h`:

| Timing | Rule | Meaning |
|---|---|---|
| `before` | `D + 1 day <= T` | the whole calendar day ended before `T`: the procedure certainly preceded the prediction point |
| `after` | `D >= T` | the day began at or after `T`: the procedure certainly followed it |
| `ambiguous` | otherwise | the day contains `T`; the order cannot be determined |

Each SMM admission is then assigned **one** category: **A** at least one procedure indicator certainly before `T`; **B** none certainly before but at
least one ambiguous; **C** procedure indicators exist and all are certainly after `T`; **D** no procedure indicator (the SMM label rests on diagnosis codes only; no time available).
Admissions are also split by whether any diagnosis-based indicator is present (its time is unknown).

**Also computed (consistency checks).** The 42 delivery hospitalizations excluded because they ended before 24 h (expected 2 SMM, 1 death), and the timing of the 10 in-hospital
deaths of the obstetric universe relative to `T` (expected 5 before `T`).

**Data-use rules.** Only counts are printed; the only file written is `data/outputs_smm_timing.txt` (published with the code as aggregate counts and metrics).
No row, identifier or date is displayed or saved. Nothing is written to the database (read-only `SELECT`s).

**Run order.** Run all cells top to bottom (seconds to a few minutes).

### **1. Libraries and connection**

In [ ]:
import os, sys, io, contextlib
import pandas as pd
from sqlalchemy import text

# notebooks/ -> project root is one level up
sys.path.append(os.path.abspath(os.path.join("..")))
import libraries.database as db_lib
engine = db_lib.get_connect_database()

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

EXPECTED = {"eligible_smm": 445, "excluded_deliveries": 42, "excluded_smm": 2, "excluded_deaths": 1, "deaths_obstetric": 10, "deaths_before_T": 5}

def q(sql):
    # text() is required: the queries contain a literal percent sign (LIKE ... || '%'), which the
    # PostgreSQL driver would otherwise read as a parameter placeholder ("immutabledict is not a sequence").
    return pd.read_sql(text(sql), engine)

### **2. Classification of the procedure-based SMM indicators (admission level)**

The query builds, for the eligible SMM deliveries, one row per procedure-based indicator hit with its timing, and one row per admission with its category.
Only the aggregated tables are fetched.

In [ ]:
BASE = """
WITH base AS (
    SELECT hadm_id, admittime, admittime + INTERVAL '24 hours' AS t_pred
    FROM mimiciv.predictors_24h_v2
    WHERE eligible_24h AND is_delivery_v2 AND smm_any
),
pr AS (
    SELECT b.hadm_id, p.indicator_name,
           CASE WHEN CAST(r.chartdate AS TIMESTAMP) + INTERVAL '1 day' <= b.t_pred THEN 'before'
                WHEN CAST(r.chartdate AS TIMESTAMP) >= b.t_pred THEN 'after'
                ELSE 'ambiguous' END AS timing
    FROM base b
    JOIN mimiciv.procedures_icd r ON r.hadm_id = b.hadm_id
    JOIN mimiciv.smm_code_patterns_v2 p
      ON p.code_type = 'PR' AND p.icd_version = r.icd_version
     AND ( (p.match_type = 'PREFIX' AND r.icd_code LIKE p.code_pattern || '%')
        OR (p.match_type = 'EXACT'  AND r.icd_code = p.code_pattern) )
),
dx AS (
    SELECT DISTINCT b.hadm_id
    FROM base b
    JOIN mimiciv.diagnoses_icd d ON d.hadm_id = b.hadm_id
    JOIN mimiciv.smm_code_patterns_v2 p
      ON p.code_type = 'DX' AND p.icd_version = d.icd_version
     AND ( (p.match_type = 'PREFIX' AND d.icd_code LIKE p.code_pattern || '%')
        OR (p.match_type = 'EXACT'  AND d.icd_code = p.code_pattern) )
),
adm AS (
    SELECT b.hadm_id,
           COUNT(pr.timing)                                          AS n_pr,
           SUM(CASE WHEN pr.timing = 'before'    THEN 1 ELSE 0 END)  AS n_before,
           SUM(CASE WHEN pr.timing = 'ambiguous' THEN 1 ELSE 0 END)  AS n_amb,
           SUM(CASE WHEN pr.timing = 'after'     THEN 1 ELSE 0 END)  AS n_after,
           CASE WHEN dx.hadm_id IS NULL THEN 0 ELSE 1 END            AS has_dx
    FROM base b
    LEFT JOIN pr ON pr.hadm_id = b.hadm_id
    LEFT JOIN dx ON dx.hadm_id = b.hadm_id
    GROUP BY b.hadm_id, dx.hadm_id
)
"""

SQL_CATEGORY = BASE + """
SELECT category, has_dx, COUNT(*) AS admissions
FROM (
    SELECT has_dx,
           CASE WHEN n_pr = 0     THEN 'D. no procedure indicator (diagnosis codes only; no time)'
                WHEN n_before > 0 THEN 'A. a procedure indicator certainly before T'
                WHEN n_amb    > 0 THEN 'B. none certainly before T, at least one ambiguous'
                ELSE                   'C. procedure indicators, all certainly after T' END AS category
    FROM adm
) t
GROUP BY category, has_dx
ORDER BY category, has_dx
"""

SQL_INDICATOR = BASE + """
SELECT indicator_name, timing, COUNT(DISTINCT hadm_id) AS admissions
FROM pr
GROUP BY indicator_name, timing
ORDER BY indicator_name, timing
"""

SQL_INDICATOR_DISTINCT = BASE + """
SELECT indicator_name, COUNT(DISTINCT hadm_id) AS distinct_admissions
FROM pr
GROUP BY indicator_name
"""

SQL_TOTAL = BASE + "SELECT COUNT(*) AS smm_admissions FROM adm"

In [ ]:
total = int(q(SQL_TOTAL).smm_admissions.iloc[0])
cat = q(SQL_CATEGORY)
cat["has_dx"] = cat["has_dx"].astype(int)
ind = q(SQL_INDICATOR)
ind_distinct = q(SQL_INDICATOR_DISTINCT).set_index("indicator_name")["distinct_admissions"]

def report_category():
    print("=" * 110)
    print(f"SMM admissions among eligible delivery hospitalizations: {total}  (expected {EXPECTED['eligible_smm']}) "
          f"[{'OK' if total == EXPECTED['eligible_smm'] else 'CHECK: differs from the audit counts'}]")
    print("T = admission time + 24 h; timing from the calendar date of each procedure-based SMM indicator (see the method table)")
    print("=" * 110)
    wide = cat.pivot_table(index="category", columns="has_dx", values="admissions", aggfunc="sum", fill_value=0)
    wide = wide.rename(columns={0: "no diagnosis-based indicator", 1: "with a diagnosis-based indicator"})
    wide["total"] = wide.sum(axis=1)
    wide["% of SMM"] = (100 * wide["total"] / total).round(1)
    print("\n[1] Category of every SMM admission (A to D); the last columns split by presence of a diagnosis-based indicator")
    print(wide.to_string())
    print(f"    Sum of categories: {int(wide['total'].sum())}  [{'OK' if int(wide['total'].sum()) == total else 'CHECK'}]")
    a = int(wide.loc[wide.index.str.startswith("A"), "total"].sum()) if wide.index.str.startswith("A").any() else 0
    b = int(wide.loc[wide.index.str.startswith("B"), "total"].sum()) if wide.index.str.startswith("B").any() else 0
    c = int(wide.loc[wide.index.str.startswith("C"), "total"].sum()) if wide.index.str.startswith("C").any() else 0
    d_ = int(wide.loc[wide.index.str.startswith("D"), "total"].sum()) if wide.index.str.startswith("D").any() else 0
    print("\n    Reading: at least %d SMM admissions (%.1f%%) had a procedure indicator certainly before T; %d more (%.1f%%) cannot be ordered;"
          % (a, 100 * a / total, b, 100 * b / total))
    print("    %d (%.1f%%) have procedure indicators only after T; %d (%.1f%%) rest on diagnosis codes alone and cannot be placed in time."
          % (c, 100 * c / total, d_, 100 * d_ / total))
    pw = ind.pivot_table(index="indicator_name", columns="timing", values="admissions", aggfunc="sum", fill_value=0)
    for t in ("before", "ambiguous", "after"):
        if t not in pw.columns:
            pw[t] = 0
    pw = pw[["before", "ambiguous", "after"]]
    pw["distinct admissions with the indicator"] = ind_distinct.reindex(pw.index).astype(int)
    print("\n[2] Procedure-based indicators: admissions by timing of the indicator (an admission with the same indicator on different days can appear in")
    print("    several of the first three columns, so they need not add up to the last column, which counts each admission once)")
    print(pw.sort_values("distinct admissions with the indicator", ascending=False).to_string())

report_category()

### **3. Consistency checks with counts already certified**

(a) Delivery hospitalizations excluded because the stay ended before 24 h; (b) timing of the in-hospital deaths of the obstetric universe relative to `T`.

In [ ]:
SQL_EXCL = """
SELECT COUNT(*) AS n,
       SUM(CASE WHEN smm_any THEN 1 ELSE 0 END) AS smm,
       SUM(CASE WHEN died_same_admission THEN 1 ELSE 0 END) AS deaths
FROM mimiciv.predictors_24h_v2
WHERE is_delivery_v2 AND NOT COALESCE(eligible_24h, FALSE)
"""
SQL_DEATH = """
SELECT CASE WHEN a.deathtime IS NULL THEN 'death without recorded time'
            WHEN a.deathtime <  p.admittime + INTERVAL '24 hours' THEN 'before T (stay ended before the prediction point)'
            ELSE 'at or after T' END AS timing,
       CASE WHEN p.is_delivery_v2 THEN 'delivery' ELSE 'other obstetric' END AS admission_type,
       COUNT(*) AS deaths
FROM mimiciv.predictors_24h_v2 p
JOIN mimiciv.admissions a ON a.hadm_id = p.hadm_id
WHERE p.died_same_admission
GROUP BY 1, 2
ORDER BY 1, 2
"""
ex = q(SQL_EXCL)
dt = q(SQL_DEATH)

def report_checks():
    print("=" * 110)
    print("CONSISTENCY CHECKS")
    print("=" * 110)
    n, s, d = int(ex.n.iloc[0]), int(ex.smm.iloc[0] or 0), int(ex.deaths.iloc[0] or 0)
    print(f"\n[3] Delivery hospitalizations not eligible at 24 h: {n} (expected {EXPECTED['excluded_deliveries']}), "
          f"with SMM {s} (expected {EXPECTED['excluded_smm']}), with death {d} (expected {EXPECTED['excluded_deaths']})  "
          f"[{'OK' if (n, s, d) == (EXPECTED['excluded_deliveries'], EXPECTED['excluded_smm'], EXPECTED['excluded_deaths']) else 'CHECK'}]")
    print("\n[4] In-hospital deaths of the obstetric universe by timing relative to T")
    print(dt.to_string(index=False))
    tot = int(dt.deaths.sum()); bef = int(dt.loc[dt.timing.str.startswith("before"), "deaths"].sum())
    print(f"    Deaths: {tot} (expected {EXPECTED['deaths_obstetric']}); before T: {bef} (expected {EXPECTED['deaths_before_T']})  "
          f"[{'OK' if (tot, bef) == (EXPECTED['deaths_obstetric'], EXPECTED['deaths_before_T']) else 'CHECK'}]")

report_checks()

### **4. Save the aggregate report**

Writes `data/outputs_smm_timing.txt` (text with the tables printed above; counts only).

In [ ]:
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    report_category()
    print()
    report_checks()
text = buf.getvalue()
with open("../data/outputs_smm_timing.txt", "w", encoding="utf-8") as fh:
    fh.write(text)
print(f"[aggregate report saved as data/outputs_smm_timing.txt  ({len(text)} characters)]")